# RAG on the GDPR: bi-encoder retrieval vs cross-encoder re-ranking

**Document:** Regulation (EU) 2016/679 (GDPR), official EUR-Lex PDF, 88 pages.

I picked the GDPR because it has two parts that talk about the same things in the same words:
173 recitals, which explain the background and are not binding, and 99 articles, which contain the actual rules.
Many articles also put a rule in one paragraph and its exceptions ("Paragraph 1 shall not apply if...") in another.
That makes it a good document for checking whether retrieval finds the passage that answers the question, or just a passage on the same topic.

Sections:
1. Text extraction, cleaning and chunking
2. Models, FAISS index and Gemini
3. Pipeline 1 (bi-encoder top 3) and Pipeline 2 (cross-encoder re-ranked top 3)
4. Failure queries and rank shifts
5. Latency
6. Scaling: cost of running the cross-encoder on every chunk

## 1. Text extraction and chunking

In [1]:
import re
import pymupdf          # PyMuPDF: gives the text of each line along with its font size
import pandas as pd

pd.set_option("display.max_colwidth", 120)
PDF_PATH = "gdpr.pdf"

### 1.1 Reading the PDF

Each page of the Official Journal has the same header lines (`L 119/34`, `EN`, `Official Journal of the European Union`, `4.5.2016`) and footnotes at the bottom.
Body text is set in 9.6 pt and footnotes in 8.5 pt, so I drop footnotes by font size and the header lines by matching them.

I first tried removing footnotes by their text (lines starting with "(1) OJ ..." or "(4) Directive ..."). That broke twice: a footnote "(4) Directive 95/46/EC" got read as recital 4, and the real recital 171 starts with "Directive 95/46/EC" and got deleted. Font size does not have this problem.

In [2]:
# Lines that repeat on every page and carry no content
HEADER_PATTERNS = [
    r"^L 119/\d+$",                          # page number in the journal, e.g. "L 119/34"
    r"^EN$",                                 # language tag
    r"^Official Journal of the European Union$",
    r"^4\.5\.2016$",                         # publication date
]
header_re = re.compile("|".join(HEADER_PATTERNS))

BODY_MIN_FONT = 9.0   # body text is 9.6 pt, footnotes are 8.5 pt; anything smaller than 9 is not body text

def page_lines(doc):
    """Go through every page and return (page_number, line_text) for body-text lines only."""
    out = []
    for pno, page in enumerate(doc, start=1):
        # "dict" mode gives blocks -> lines -> spans, and each span has its font size
        for block in page.get_text("dict")["blocks"]:
            for line in block.get("lines", []):          # image blocks have no "lines" key
                spans = line["spans"]
                if not spans or max(s["size"] for s in spans) < BODY_MIN_FONT:
                    continue                              # footnote / small print
                text = "".join(s["text"] for s in spans)
                text = text.replace("\xad", "").strip()   # \xad is a soft hyphen left over from words split across lines
                if not text or header_re.match(text):
                    continue
                out.append((pno, text))
    return out

doc = pymupdf.open(PDF_PATH)
lines = page_lines(doc)
print(f"{doc.page_count} pages, {len(lines)} body-text lines kept")

88 pages, 4124 body-text lines kept


### 1.2 Splitting into recitals and articles

Before chunking, I cut the text at its natural boundaries, so that no chunk mixes the end of one article with the start of the next.
- A recital starts with `(n)`. Up to recital 99 the number is on a line by itself; from 100 onwards it is on the same line as the text. Recital numbers also have to come in order (1, 2, 3 ...), otherwise list items like `(2)` inside a recital would be taken as new recitals.
- An article starts with a line `Article n`, and the line after it is the article's title.
- `CHAPTER X` and `Section n` headings, and the title line below them, are skipped.

In [3]:
article_re = re.compile(r"^Article (\d+)$")             # "Article 17"
recital_re = re.compile(r"^\((\d+)\)\s*(.*)$")          # "(85)" alone, or "(100) In order to ..."
heading_re = re.compile(r"^(CHAPTER [IVXL]+|Section \d+)$")

units = []               # one entry per recital or article
current = None           # the recital/article currently being filled with lines
in_articles = False      # the document has all recitals first, then all articles
expected_recital = 1     # next recital number we should see
skip_next = False        # set when the next line is a title we do not want in the text

for i, (pno, line) in enumerate(lines):
    if skip_next:
        skip_next = False
        continue

    if heading_re.match(line):            # chapter/section heading: drop it and the title under it
        skip_next = True
        continue

    m_art = article_re.match(line)
    if m_art:
        in_articles = True
        title = lines[i + 1][1]           # e.g. "Right to erasure ('right to be forgotten')"
        current = {"part": "article", "section": f"Article {m_art.group(1)} - {title}",
                   "page": pno, "lines": []}
        units.append(current)
        skip_next = True                  # the title is kept in "section", not repeated in the text
        continue

    m_rec = recital_re.match(line)
    if not in_articles and m_rec and int(m_rec.group(1)) == expected_recital:
        expected_recital += 1
        current = {"part": "recital", "section": f"Recital ({m_rec.group(1)})",
                   "page": pno, "lines": []}
        if m_rec.group(2):                # from recital 100 on, text starts on the same line
            current["lines"].append(m_rec.group(2))
        units.append(current)
        continue

    if current is not None:               # anything before Recital (1) is the preamble and is ignored
        current["lines"].append(line)

# join the lines of each unit into one string with single spaces
for u in units:
    u["text"] = re.sub(r"\s+", " ", " ".join(u.pop("lines"))).strip()

n_rec = sum(u["part"] == "recital" for u in units)
n_art = sum(u["part"] == "article" for u in units)
print(f"recitals: {n_rec}   articles: {n_art}")
assert n_rec == 173 and n_art == 99, "split did not find every recital/article"

recitals: 173   articles: 99


### 1.3 Chunking

A recital or article of up to 200 words becomes one chunk. Longer ones are cut into 200-word windows that overlap by 40 words, so a sentence cut at the edge of one window is still complete in the next one.
200 words is about 260 tokens, which is below the 512-token input limit of both MiniLM models, so no chunk gets cut off silently by the models.
Every chunk keeps its section name and page number, so any result can be traced back to the PDF.

In [4]:
CHUNK_WORDS = 200    # max words per chunk
OVERLAP = 40         # words shared between two neighbouring windows of the same article

chunks = []
for u in units:
    words = u["text"].split()
    start = 0
    while True:
        piece = words[start:start + CHUNK_WORDS]
        chunks.append({
            "chunk_id": len(chunks),      # chunk_id is also the row index of chunks_df
            "part": u["part"],
            "section": u["section"],
            "page": u["page"],
            "n_words": len(piece),
            "text": " ".join(piece),
        })
        if start + CHUNK_WORDS >= len(words):   # this window reached the end of the unit
            break
        start += CHUNK_WORDS - OVERLAP          # next window starts 160 words later

chunks_df = pd.DataFrame(chunks)
print("total chunks:", len(chunks_df))
print(chunks_df.groupby("part").size().to_string())
chunks_df["n_words"].describe().round(1)

total chunks: 434
part
article    220
recital    214


count    434.0
mean     137.1
std       60.4
min       22.0
25%       77.2
50%      142.0
75%      200.0
max      200.0
Name: n_words, dtype: float64

In [5]:
# Print a recital, the start of Article 9 and the last chunk to check the cleaning by eye
sample_ids = [0,
              chunks_df.index[chunks_df.section.str.startswith("Article 9 ")][0],
              len(chunks_df) - 1]
for cid in sample_ids:
    r = chunks_df.loc[cid]
    print(f"[chunk {cid}] {r.section} (page {r.page}, {r.n_words} words)\n{r.text[:400]}...\n")

[chunk 0] Recital (1) (page 1, 61 words)
The protection of natural persons in relation to the processing of personal data is a fundamental right. Article 8(1) of the Charter of Fundamental Rights of the European Union (the ‘Charter’) and Article 16(1) of the Treaty on the Functioning of the European Union (TFEU) provide that everyone has the right to the protection of personal data concerning him or her....

[chunk 235] Article 9 - Processing of special categories of personal data (page 38, 200 words)
1. Processing of personal data revealing racial or ethnic origin, political opinions, religious or philosophical beliefs, or trade union membership, and the processing of genetic data, biometric data for the purpose of uniquely identifying a natural person, data concerning health or data concerning a natural person's sex life or sexual orientation shall be prohibited. 2. Paragraph 1 shall not appl...

[chunk 433] Article 99 - Entry into force and application (page 87, 64 words)
1. This Re

In [6]:
# Keep the chunks in a file so any chunk id mentioned later can be looked up
chunks_df.to_csv("chunks.csv", index=False)

434 chunks: 214 from recitals and 220 from articles. Article 9 shows why this document is useful: paragraph 1 says processing of health, biometric etc. data "shall be prohibited", and paragraph 2 starts the list of exceptions. Long articles like this end up with the rule and its exceptions in different chunks.

## 2. Models, vector index and the LLM

- Bi-encoder: `sentence-transformers/all-MiniLM-L6-v2`. Turns any text into one 384-dimensional vector. All chunks are encoded once and stored in FAISS.
- Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`, trained for passage ranking on MS MARCO. It reads the query and one chunk together and gives a single relevance score.
- LLM: Gemini through the `google-genai` SDK, temperature 0, so the two answers for a query can only differ because they got different passages.

Both encoders run on the CPU of my MacBook Air (M1) so the latency numbers are repeatable.

In [7]:
import os, time, json
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"      # torch and faiss each bring their own OpenMP library; on macOS this stops the "duplicate library" abort
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # avoids a tokenizer warning/deadlock inside Jupyter

import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss                                      # imported after torch (import order matters for the OpenMP clash on macOS)
faiss.omp_set_num_threads(1)                      # multi-threaded FAISS search crashed the kernel next to torch; one thread is stable

DEVICE = "cpu"
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE)
print("models loaded on", DEVICE, "| torch", torch.__version__, "| faiss", faiss.__version__)

W1019 13:39:33.092000 33632 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

models loaded on cpu | torch 2.14.1 | faiss 1.15.1


### 2.1 Indexing the chunks (bi-encoder + FAISS)

The embeddings are normalised to length 1, so the inner product that `IndexFlatIP` computes is the same as cosine similarity.
`IndexFlatIP` compares the query with every stored vector (exact search). With 434 chunks an approximate index is not needed, and exact search means any ranking mistake comes from the embedding model and not from the index.

In [8]:
t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode(
    chunks_df["text"].tolist(),
    batch_size=32,
    normalize_embeddings=True,      # length-1 vectors: inner product = cosine similarity
    convert_to_numpy=True,
    show_progress_bar=True,
).astype("float32")                 # FAISS works with float32

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])   # 384 = embedding size
index.add(chunk_embeddings)                            # row i of the index = chunk_id i
index_build_seconds = time.perf_counter() - t0

print(f"index holds {index.ntotal} vectors of dim {chunk_embeddings.shape[1]}, built in {index_build_seconds:.1f} s")

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

index holds 434 vectors of dim 384, built in 4.7 s


### 2.2 Gemini

The API key is read from a local `.env` file. That file is in `.gitignore`, so the key is not in the repository.

In [9]:
from google import genai
from google.genai import types

def read_env(path=".env"):
    """Read KEY=value lines from the .env file into a dict."""
    env = {}
    with open(path) as f:
        for line in f:
            if "=" in line and not line.startswith("#"):
                k, v = line.strip().split("=", 1)
                env[k] = v
    return env

client = genai.Client(api_key=read_env()["GEMINI_API_KEY"])

# Flash models available to this key (gemini-2.5-flash is listed but no longer open to new keys)
flash_models = [m.name for m in client.models.list() if "flash" in m.name]
print("\n".join(flash_models))

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


The free tier has a small daily request limit per model, and I ran out on `gemini-3.8-flash` while testing.
So the code below tries a short list of Flash models in order and moves to the next one when a model's daily quota is used up.
Both answers of the same query always come from the same model, and every answer is stored together with the model name.

Answers are also saved in `results/llm_cache.json`, keyed by the full prompt. Running the notebook again gives back the same answers without new API calls.

In [10]:
import hashlib

LLM_CANDIDATES = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.5-flash", "gemini-3.1-flash-lite"]
LLM_NAME = LLM_CANDIDATES[0]      # model currently in use
exhausted_models = set()          # models whose daily quota ran out in this session

BASE_CONFIG = dict(temperature=0.0, max_output_tokens=2048)   # 2048 leaves room in case a model insists on "thinking" tokens

def make_config(thinking_off=True):
    """Generation settings. thinking_budget=0 turns off Gemini's hidden reasoning step,
    so the answer is written straight from the passages (also faster)."""
    if thinking_off:
        return types.GenerateContentConfig(**BASE_CONFIG,
                                           thinking_config=types.ThinkingConfig(thinking_budget=0))
    return types.GenerateContentConfig(**BASE_CONFIG)

GEN_CONFIG = make_config(thinking_off=True)

os.makedirs("results", exist_ok=True)
CACHE_PATH = "results/llm_cache.json"
llm_cache = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}
LAST_MODEL = None                 # model that produced the most recent answer

def _generate(prompt):
    """Send one prompt to the current model.
    - model refuses thinking_budget=0      -> retry with its default setting
    - daily quota used up (429 per day)    -> switch to the next model in LLM_CANDIDATES
    - per-minute limit or server overload  -> wait and retry"""
    global GEN_CONFIG, LLM_NAME
    attempt = 0
    while True:
        try:
            resp = client.models.generate_content(model=LLM_NAME, contents=prompt, config=GEN_CONFIG)
            return (resp.text or "").strip()
        except Exception as e:
            msg = str(e)
            if "thinking" in msg.lower() and GEN_CONFIG.thinking_config is not None:
                print(f"{LLM_NAME} does not allow thinking to be switched off; using its default")
                GEN_CONFIG = make_config(thinking_off=False)
                continue
            quota = "429" in msg or "RESOURCE_EXHAUSTED" in msg
            daily = "perday" in msg.lower().replace(" ", "") or "daily" in msg.lower()
            if quota and (daily or attempt >= 2):          # 429 that does not clear after waiting = daily limit
                exhausted_models.add(LLM_NAME)
                left = [m for m in LLM_CANDIDATES if m not in exhausted_models]
                print(f"{LLM_NAME} out of quota: {msg[:160]}")
                if not left:
                    raise RuntimeError("every candidate model is out of quota; free-tier quota resets at midnight US Pacific time")
                LLM_NAME, GEN_CONFIG, attempt = left[0], make_config(thinking_off=True), 0
                print("switching to", LLM_NAME)
                continue
            if (quota or any(c in msg for c in ("500", "503", "UNAVAILABLE", "INTERNAL"))) and attempt < 4:
                wait = 20 * (attempt + 1)
                print(f"{LLM_NAME}: {msg[:120]} ... waiting {wait}s")
                time.sleep(wait)
                attempt += 1
                continue
            raise

def call_llm(prompt, use_cache=True):
    """Return Gemini's answer to a prompt. Uses the saved answer if this exact prompt was asked before
    (unless use_cache=False). LAST_MODEL is set to the model that produced the answer."""
    global LAST_MODEL
    key = hashlib.sha256(prompt.encode()).hexdigest()   # same prompt -> same key
    if use_cache and key in llm_cache:
        LAST_MODEL = llm_cache[key]["model"]
        return llm_cache[key]["answer"]
    answer = _generate(prompt)
    LAST_MODEL = LLM_NAME
    llm_cache[key] = {"model": LLM_NAME, "answer": answer}
    with open(CACHE_PATH, "w") as f:
        json.dump(llm_cache, f, indent=1)
    return answer

print("starting model:", LLM_NAME, "| saved answers:", len(llm_cache))

starting model: gemini-3.8-flash | saved answers: 14


## 3. The two pipelines

Both pipelines use the same bi-encoder top 10 and send exactly 3 chunks to Gemini with the same prompt. The only difference is which 3:
- **Pipeline 1 (baseline):** the first 3 by cosine similarity.
- **Pipeline 2 (re-ranked):** the cross-encoder scores all 10 (query, chunk) pairs, the 10 are sorted by that score (highest first), and the first 3 are used.

In [11]:
TOP_K = 10      # candidates taken from the bi-encoder
TOP_N = 3       # chunks given to the LLM

def baseline_retrieve(query, k=TOP_K):
    """Pipeline 1 retrieval: embed the query, get the k most similar chunks from FAISS (best first)."""
    q_emb = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q_emb, k)               # arrays of shape (1, k): cosine scores and chunk ids
    cands = chunks_df.loc[ids[0], ["chunk_id", "part", "section", "page"]].copy()
    cands["bi_score"] = scores[0]
    cands["bi_rank"] = np.arange(1, k + 1)             # 1 = most similar
    return cands.reset_index(drop=True)

def rerank(query, cands):
    """Pipeline 2 re-ranking: the cross-encoder reads (query, chunk text) as one input for each
    candidate, gives a relevance score, and the candidates are sorted by that score."""
    pairs = [(query, chunks_df.at[cid, "text"]) for cid in cands["chunk_id"]]
    ce_scores = cross_encoder.predict(pairs, batch_size=TOP_K, show_progress_bar=False)   # all 10 pairs in one batch
    out = cands.copy()
    out["ce_score"] = ce_scores                        # raw score (logit); higher = more relevant
    out = out.sort_values("ce_score", ascending=False).reset_index(drop=True)
    out["ce_rank"] = np.arange(1, len(out) + 1)
    return out

### 3.1 Prompt (the same for both pipelines)

The prompt allows only the given passages and asks the model to say what is missing instead of guessing.
That way, when the right chunk is missing, it shows up as "the passages do not say" or as an answer that leaves out the exception, and I can trace it back to retrieval.
Each passage is labelled with its article or recital so the answer can cite it.

In [12]:
PROMPT_TEMPLATE = """You are answering questions about the EU General Data Protection Regulation (GDPR).
Use ONLY the context passages below. Do not use outside knowledge.
If the passages do not contain enough information to answer, say what is missing instead of guessing.
Mention the Article or Recital each part of your answer comes from.

Context:
{context}

Question: {question}

Answer (at most 150 words):"""

def build_prompt(query, chunk_ids):
    """Fill the template with the given chunks, in the given order, each with its section label."""
    blocks = []
    for i, cid in enumerate(chunk_ids, start=1):
        row = chunks_df.loc[cid]
        blocks.append(f"[Passage {i} | {row.section}]\n{row.text}")
    return PROMPT_TEMPLATE.format(context="\n\n".join(blocks), question=query)

In [13]:
def run_both(query, show=True):
    """Run Pipeline 1 and Pipeline 2 for one query and collect ranks, contexts, answers and timings."""
    t = {}

    t0 = time.perf_counter()
    cands = baseline_retrieve(query)                   # top 10 by cosine
    t["bi_retrieval_ms"] = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    reranked = rerank(query, cands)                    # same 10, re-ordered by the cross-encoder
    t["cross_encoder_ms"] = (time.perf_counter() - t0) * 1000

    p1_ids = cands["chunk_id"].tolist()[:TOP_N]        # Pipeline 1 context
    p2_ids = reranked["chunk_id"].tolist()[:TOP_N]     # Pipeline 2 context

    t0 = time.perf_counter()
    answer1 = call_llm(build_prompt(query, p1_ids))
    model1 = LAST_MODEL
    t["llm_p1_ms"] = (time.perf_counter() - t0) * 1000   # close to 0 when the answer came from the saved cache

    if p2_ids == p1_ids:
        # same 3 chunks in the same order -> identical prompt at temperature 0, so the answer is the same
        answer2, model2 = answer1, model1
        t["llm_p2_ms"] = t["llm_p1_ms"]
    else:
        t0 = time.perf_counter()
        answer2 = call_llm(build_prompt(query, p2_ids))
        model2 = LAST_MODEL
        t["llm_p2_ms"] = (time.perf_counter() - t0) * 1000
        if model2 != model1:
            # quota ran out between the two calls and the model changed: ask Answer 1 again on the new model
            answer1 = call_llm(build_prompt(query, p1_ids), use_cache=False)
            model1 = LAST_MODEL

    # one row per candidate: its rank by the bi-encoder and its rank after re-ranking
    table = reranked[["chunk_id", "section", "bi_rank", "ce_rank", "bi_score", "ce_score"]].copy()
    table["shift"] = table["bi_rank"] - table["ce_rank"]          # positive = moved up after re-ranking
    table = table.sort_values("bi_rank").reset_index(drop=True)

    result = {"query": query, "table": table, "p1_ids": p1_ids, "p2_ids": p2_ids,
              "answer1": answer1, "answer2": answer2, "model": model1, "timing": t}
    if show:
        show_result(result)
    return result

def show_result(r):
    """Print the rank table (rows highlighted = Pipeline 2's top 3), both contexts and both answers."""
    print("QUERY:", r["query"])
    display(r["table"].style.format({"bi_score": "{:.3f}", "ce_score": "{:.2f}"})
            .apply(lambda row: ["background-color: #fff3c4" if row.ce_rank <= TOP_N else "" for _ in row], axis=1))
    print("P1 context:", [chunks_df.at[c, "section"] for c in r["p1_ids"]])
    print("P2 context:", [chunks_df.at[c, "section"] for c in r["p2_ids"]])
    print("\n--- Answer 1 (baseline) ---\n" + r["answer1"])
    print("\n--- Answer 2 (re-ranked) ---\n" + r["answer2"])
    print("\nLLM:", r["model"], "| timing (ms):", {k: round(v, 1) for k, v in r["timing"].items()})

### 3.2 A simple question first

The table is in bi-encoder order. `ce_rank` is the position after re-ranking, and the highlighted rows are the 3 chunks Pipeline 2 sends to Gemini.

In [14]:
r = run_both("Within how many hours must a controller notify the supervisory authority of a personal data breach?")

QUERY: Within how many hours must a controller notify the supervisory authority of a personal data breach?


,chunk_id,section,bi_rank,ce_rank,bi_score,ce_score,shift
0,109,Recital (85),1,2,0.781,8.40,-1
1,291,Article 33 - Notification of a personal data breach to the supervisory authority,2,1,0.770,9.06,1
2,111,Recital (87),3,5,0.700,1.60,-2
3,110,Recital (86),4,4,0.671,2.07,0
4,292,Article 33 - Notification of a personal data breach to the supervisory authority,5,7,0.641,1.07,-2
5,300,Article 36 - Prior consultation,6,3,0.640,3.07,3
6,294,Article 34 - Communication of a personal data breach to the data subject,7,9,0.582,0.20,-2
7,251,Article 14 - Information to be provided where personal data have not been obtained from the data subject,8,8,0.560,0.71,0
8,293,Article 34 - Communication of a personal data breach to the data subject,9,10,0.554,-0.13,-1
9,243,"Article 12 - Transparent information, communication and modalities for the exercise of the rights of the data",10,6,0.551,1.46,4


P1 context: ['Recital (85)', 'Article 33 - Notification of a personal data breach to the supervisory authority', 'Recital (87)']
P2 context: ['Article 33 - Notification of a personal data breach to the supervisory authority', 'Recital (85)', 'Article 36 - Prior consultation']

--- Answer 1 (baseline) ---
A controller must notify the supervisory authority of a personal data breach without undue delay and, where feasible, not later than **72 hours** after having become aware of it (Article 33(1), Recital (85)). 

This notification is required unless the personal data breach is unlikely to result in a risk to the rights and freedoms of natural persons (Article 33(1), Recital (85)). If the notification cannot be achieved or made within 72 hours, it must be accompanied by the reasons for the delay (Article 33(1), Recital (85)).

--- Answer 2 (re-ranked) ---
Based on **Article 33(1)** and **Recital (85)**, the controller must notify the competent supervisory authority of a personal data brea

Both answers say 72 hours, because both contexts contain Article 33. The ranking still differs: the bi-encoder puts Recital 85 at #1, above Article 33 itself. Recital 85 explains the same rule in nearly the same words but is not the binding text. The cross-encoder puts Article 33 first. This recital-above-article pattern comes back in the failure cases below.

## 4. Failure-mode analysis

How I set this up:
1. I wrote 29 questions in four groups: keyword trap / synonyms, exceptions, negation, and recital vs article (GDPR-specific).
2. For each question I wrote down a **gold phrase**: a short exact phrase that only the passage answering the question contains. The cell checks that every phrase exists in `chunks_df`. The "gold chunk" is whichever chunk contains the phrase.
3. For every question I run retrieval only (no LLM) and record the gold chunk's rank before and after re-ranking.
4. Gemini is then called only for questions where the gold chunk moved into or out of the top 3, because those are the only cases where the two answers can differ.

Questions are written the way a normal person would ask them, which is often not the wording the regulation uses.

In [15]:
# Each entry: id, group, question, gold phrase, optional "section" (used when the phrase also appears
# in a recital and only the article counts), and a short note on what makes the question hard.
FAILURE_QUERIES = [
    # keyword trap / synonyms: everyday words, while the regulation uses its own terms
    dict(qid="K1", category="keyword trap", query="Can I make a company delete everything it knows about me?",
         gold="have the right to obtain from the controller the erasure", note="'delete' vs 'erasure'"),
    dict(qid="K2", category="keyword trap", query="Can my job application be rejected purely by a computer program without a human looking at it?",
         gold="not to be subject to a decision based solely on automated processing", note="'computer program' vs 'automated processing'"),
    dict(qid="K3", category="keyword trap", query="What is the maximum penalty a company can get for violating the regulation?",
         gold="20 000 000 EUR", note="'penalty' vs 'administrative fine'; Art 84 is titled 'Penalties' but is about national penalties"),
    dict(qid="K4", category="keyword trap", query="Can I take my data with me when I switch to a different service provider?",
         gold="in a structured, commonly used and machine-readable format", note="'take my data with me' vs 'data portability'"),
    dict(qid="K5", category="keyword trap", query="How long can a company take to reply when I ask to see my data?",
         gold="extended by two further months", note="'reply' vs 'provide information on action taken'"),

    # exceptions: the general rule is in one place, the "except when" condition in another
    dict(qid="E1", category="exception", query="Does a company always have to inform affected people about a data breach?",
         gold="shall not be required if any of the following conditions are met", note="Art 34(3) exemptions are in a different chunk from the duty in 34(1)"),
    dict(qid="E2", category="exception", query="Is a company always required to erase my data when I ask?",
         gold="Paragraphs 1 and 2 shall not apply to the extent that processing is necessary", note="Art 17(3) exceptions are in the second chunk of Art 17"),
    dict(qid="E3", category="exception", query="Does the GDPR apply to me keeping my friends' phone numbers on my personal phone?",
         gold="purely personal or household activity", section="Article 2", note="household exemption, Art 2(2)(c)"),
    dict(qid="E4", category="exception", query="Do small businesses need to maintain records of their processing activities?",
         gold="fewer than 250 persons", note="Art 30(5) size exemption at the end of the article"),
    dict(qid="E5", category="exception", query="Does every non-EU company targeting EU customers need to appoint a representative in the EU?",
         gold="processing which is occasional", note="Art 27(2) exemption"),

    # negation / constraints: "not", "cannot", "without", "prohibited"
    dict(qid="N1", category="negation", query="In which situations is processing health data not prohibited?",
         gold="Paragraph 1 shall not apply if one of the following applies", note="'not prohibited' vs the prohibition itself"),
    dict(qid="N2", category="negation", query="When can sensitive data not be processed even if the person consents?",
         gold="may not be lifted by the data subject", note="consent cannot lift the ban where national law says so"),
    dict(qid="N3", category="negation", query="What is a company not allowed to do regarding instructions to its data protection officer?",
         gold="does not receive any instructions", note="Art 38(3), DPO independence"),
    dict(qid="N4", category="negation", query="Can a processor hire a subcontractor without the controller's approval?",
         gold="shall not engage another processor without prior specific or general written authorisation", note="'subcontractor' vs 'another processor'"),
    dict(qid="N5", category="negation", query="Can a 13-year-old sign up for a social media app without parental consent?",
         gold="below the age of 16 years", note="age limit and parental consent, Art 8"),
    dict(qid="N6", category="negation", query="Can a private company keep its own register of criminal convictions?",
         gold="carried out only under the control of official authority", note="Art 10 restriction"),
    # recital vs article: recitals repeat the article's words without being the rule itself
    dict(qid="D1", category="recital vs article", query="Within how many hours must a controller notify the supervisory authority of a personal data breach?",
         gold="not later than 72 hours", section="Article 33", note="Recital 85 repeats the rule"),
    dict(qid="D2", category="recital vs article", query="When is a data protection impact assessment mandatory?",
         gold="systematic monitoring of a publicly accessible area on a large scale", note="Art 35(3) list vs Recitals 89-91"),
    dict(qid="D3", category="recital vs article", query="Which organisations must designate a data protection officer?",
         gold="regular and systematic monitoring of data subjects on a large scale", note="Art 37(1) vs Recital 97"),
    dict(qid="D4", category="recital vs article", query="Can a person withdraw consent after giving it?",
         gold="right to withdraw his or her consent at any time", note="Art 7(3) vs Recitals 42-43"),
    dict(qid="D5", category="recital vs article", query="Can fines be imposed on government bodies?",
         gold="imposed on public authorities and bodies", note="Art 83(7) vs Recital 150"),

    # more negation questions: answers are "shall not apply" / "shall not" clauses
    dict(qid="N7", category="negation", query="What types of personal data processing does the GDPR not cover?",
         gold="does not apply to the processing of personal data", section="Article 2", note="Art 2(2) vs Recitals 16/18"),
    dict(qid="N8", category="negation", query="When does a company not need to tell me that it collected my data from someone else?",
         gold="Paragraphs 1 to 4 shall not apply where and insofar as", note="Art 14(5) exemptions"),
    dict(qid="N9", category="negation", query="When can I not use my right to data portability?",
         gold="That right shall not apply to processing necessary for the performance of a task carried out in the public interest", note="Art 20(3)"),
    dict(qid="N10", category="negation", query="When is a fully automated decision about a person not forbidden?",
         gold="Paragraph 1 shall not apply if the decision", note="Art 22(2) exceptions to the ban"),
    dict(qid="N11", category="negation", query="Is no special approval needed to transfer data to a country the Commission considers adequate?",
         gold="Such a transfer shall not require any specific authorisation", note="Art 45(1)"),
    dict(qid="N12", category="negation", query="Which processing are data protection authorities not allowed to supervise?",
         gold="shall not be competent to supervise processing operations of courts", note="Art 55(3)"),
    dict(qid="N13", category="negation", query="Can the data protection officer be fired for doing their job?",
         gold="shall not be dismissed or penalised", note="'fired' vs 'dismissed or penalised'"),
    dict(qid="N14", category="negation", query="Can a processor use the data for its own purposes without the controller's instructions?",
         gold="shall not process those data except on instructions from the controller", note="Art 29"),
]

# Find the gold chunk(s) for every question; stop if a phrase is not in the corpus
for q in FAILURE_QUERIES:
    mask = chunks_df["text"].str.contains(q["gold"], regex=False)
    if "section" in q:
        mask &= chunks_df["section"].str.startswith(q["section"])
    q["gold_ids"] = chunks_df.index[mask].tolist()
    assert q["gold_ids"], f"gold phrase not found for {q['qid']}"
print(len(FAILURE_QUERIES), "queries, all gold phrases found")

29 queries, all gold phrases found


In [16]:
def gold_rank(ranked_ids, gold_ids):
    """Position (1 = top) of the first gold chunk in a ranked list of chunk ids, or None if it is not there."""
    for pos, cid in enumerate(ranked_ids, start=1):
        if cid in gold_ids:
            return pos
    return None

def outcome(bi_r, ce_r):
    """Label a question by whether the gold chunk reaches the LLM (top 3) in each pipeline."""
    in1 = bi_r is not None and bi_r <= TOP_N
    in2 = ce_r is not None and ce_r <= TOP_N
    if bi_r is None:      return "not in top 10"      # the cross-encoder never gets to see it
    if in2 and not in1:   return "fixed by re-ranking"
    if in1 and not in2:   return "hurt by re-ranking"
    if in1 and in2:       return "both correct"
    return "both miss"                                 # in the top 10 but outside the top 3 both times

rows = []
for q in FAILURE_QUERIES:
    cands = baseline_retrieve(q["query"])
    reranked = rerank(q["query"], cands)
    bi_r = gold_rank(cands["chunk_id"].tolist(), q["gold_ids"])
    ce_r = gold_rank(reranked["chunk_id"].tolist(), q["gold_ids"])
    top1_bi = cands.iloc[0]                            # what the bi-encoder thought was most relevant
    rows.append({
        "qid": q["qid"], "category": q["category"], "query": q["query"],
        "gold_bi_rank": bi_r, "gold_ce_rank": ce_r, "outcome": outcome(bi_r, ce_r),
        "bi_top1": f"{top1_bi.section[:45]} (chunk {top1_bi.chunk_id})",
    })

sweep = pd.DataFrame(rows)
sweep[["gold_bi_rank", "gold_ce_rank"]] = sweep[["gold_bi_rank", "gold_ce_rank"]].astype("Int64")   # integer column that allows missing values
sweep

,qid,category,query,gold_bi_rank,gold_ce_rank,outcome,bi_top1
0,K1,keyword trap,Can I make a company delete everything it knows about me?,4,3,fixed by re-ranking,Recital (66) (chunk 81)
1,K2,keyword trap,Can my job application be rejected purely by a computer program without a human looking at it?,<NA>,<NA>,not in top 10,Recital (67) (chunk 82)
2,K3,keyword trap,What is the maximum penalty a company can get for violating the regulation?,2,4,hurt by re-ranking,Article 84 - Penalties (chunk 414)
3,K4,keyword trap,Can I take my data with me when I switch to a different service provider?,6,5,both miss,Recital (57) (chunk 70)
4,K5,keyword trap,How long can a company take to reply when I ask to see my data?,1,1,both correct,"Article 12 - Transparent information, communi (chunk 243)"
5,E1,exception,Does a company always have to inform affected people about a data breach?,8,3,fixed by re-ranking,Recital (87) (chunk 111)
6,E2,exception,Is a company always required to erase my data when I ask?,<NA>,<NA>,not in top 10,Article 17 - Right to erasure (‘right to be f (chunk 257)
7,E3,exception,Does the GDPR apply to me keeping my friends' phone numbers on my personal phone?,<NA>,<NA>,not in top 10,Recital (18) (chunk 19)
8,E4,exception,Do small businesses need to maintain records of their processing activities?,1,3,both correct,Article 30 - Records of processing activities (chunk 287)
9,E5,exception,Does every non-EU company targeting EU customers need to appoint a representative in the EU?,<NA>,<NA>,not in top 10,Article 68 - European Data Protection Board (chunk 386)


In [17]:
# Hit@n = share of questions where the gold chunk is in the top n.
# Hit@3 is the one that matters most here, because only the top 3 reach Gemini.
# MRR@10 = mean of 1/rank of the gold chunk (0 when it is not in the top 10).
def hit_at(ranks, n):
    return np.mean([r is not pd.NA and r <= n for r in ranks])

def mrr(ranks):
    return np.mean([0 if r is pd.NA else 1 / r for r in ranks])

summary = pd.DataFrame({
    "Pipeline 1 (bi-encoder)":    [hit_at(sweep.gold_bi_rank, 1), hit_at(sweep.gold_bi_rank, 3), mrr(sweep.gold_bi_rank)],
    "Pipeline 2 (cross-encoder)": [hit_at(sweep.gold_ce_rank, 1), hit_at(sweep.gold_ce_rank, 3), mrr(sweep.gold_ce_rank)],
}, index=["Hit@1", "Hit@3", "MRR@10"]).round(3)

print(sweep["outcome"].value_counts().to_string(), "\n")
print(sweep.groupby("category")["outcome"].value_counts().unstack(fill_value=0), "\n")
summary

outcome
both correct           13
not in top 10          10
fixed by re-ranking     4
hurt by re-ranking      1
both miss               1 

outcome             both correct  both miss  fixed by re-ranking  \
category                                                           
exception                      1          0                    1   
keyword trap                   1          1                    1   
negation                       7          0                    2   
recital vs article             4          0                    0   

outcome             hurt by re-ranking  not in top 10  
category                                               
exception                            0              3  
keyword trap                         1              1  
negation                             0              5  
recital vs article                   0              1   



,Pipeline 1 (bi-encoder),Pipeline 2 (cross-encoder)
Hit@1,0.241,0.310
Hit@3,0.483,0.586
MRR@10,0.375,0.435


Re-ranking moved the gold chunk into the top 3 for 4 questions and out of it for 1. Hit@3 goes from 0.48 to 0.59 and MRR from 0.38 to 0.44.

The bigger limit is the first stage: for 10 of the 29 questions the gold chunk was not in the bi-encoder's top 10 at all, so the cross-encoder never saw it. Most of these use everyday words for something the regulation describes in legal terms (for example "subcontractor" for "another processor"), or ask for an exception buried in a long article.

### 4.1 Answers where the top 3 changed

For each question below: the rank table, the context each pipeline sent, both answers, and the rank shift of the gold chunk.

In [18]:
# Only questions where the gold chunk crossed the top-3 line in either direction
selected = sweep[sweep["outcome"].isin(["fixed by re-ranking", "hurt by re-ranking"])]["qid"].tolist()
print("running LLM for:", selected)

failure_results = {}
for qid in selected:
    q = next(x for x in FAILURE_QUERIES if x["qid"] == qid)
    print("=" * 110)
    print(f"{qid} [{q['category']}] trap: {q['note']}")
    r = run_both(q["query"])
    r["gold_ids"] = q["gold_ids"]
    failure_results[qid] = r
    g = r["table"][r["table"]["chunk_id"].isin(q["gold_ids"])].iloc[0]     # the gold chunk's row
    print(f"\nRANK SHIFT: chunk {g.chunk_id} ({g.section[:60]}) moved from #{g.bi_rank} to #{g.ce_rank}")

running LLM for: ['K1', 'K3', 'E1', 'N11', 'N13']
K1 [keyword trap] trap: 'delete' vs 'erasure'
QUERY: Can I make a company delete everything it knows about me?


,chunk_id,section,bi_rank,ce_rank,bi_score,ce_score,shift
0,81,Recital (66),1,2,0.395,-9.94,-1
1,291,Article 33 - Notification of a personal data breach to the supervisory authority,2,6,0.343,-10.80,-4
2,30,Recital (26),3,8,0.342,-11.00,-5
3,257,Article 17 - Right to erasure (‘right to be forgotten’),4,3,0.339,-10.54,1
4,45,Recital (39),5,7,0.338,-10.95,-2
5,46,Recital (39),6,4,0.337,-10.70,2
6,111,Recital (87),7,9,0.334,-11.07,-2
7,109,Recital (85),8,5,0.332,-10.71,3
8,34,Recital (29),9,10,0.331,-11.25,-1
9,79,Recital (65),10,1,0.327,-9.31,9


P1 context: ['Recital (66)', 'Article 33 - Notification of a personal data breach to the supervisory authority', 'Recital (26)']
P2 context: ['Recital (65)', 'Recital (66)', 'Article 17 - Right to erasure (‘right to be forgotten’)']

--- Answer 1 (baseline) ---
Based on the provided passages, there is no mention of a general right to make a company delete *everything* it knows about you. 

However, Recital (66) discusses a "right to erasure" (or "right to be forgotten") in the online environment. If a controller has made your personal data public, they are obliged to take reasonable steps—considering available technology and means—to inform other controllers processing that data to erase any links to, or copies or replications of, those personal data upon your request (Recital 66). 

Furthermore, data protection principles do not apply to anonymous information, as it does not relate to an identified or identifiable natural person (Recital 26). 

The context is missing specific conditio

,chunk_id,section,bi_rank,ce_rank,bi_score,ce_score,shift
0,414,Article 84 - Penalties,1,5,0.605,-1.66,-4
1,410,Article 83 - General conditions for imposing administrative fines,2,4,0.596,-1.00,-2
2,184,Recital (148),3,2,0.566,-0.16,1
3,186,Recital (150),4,3,0.518,-0.70,1
4,185,Recital (149),5,1,0.504,0.27,4
5,189,Recital (152),6,6,0.497,-3.44,0
6,408,Article 83 - General conditions for imposing administrative fines,7,7,0.482,-4.15,0
7,413,Article 83 - General conditions for imposing administrative fines,8,10,0.469,-8.27,-2
8,412,Article 83 - General conditions for imposing administrative fines,9,9,0.459,-6.64,0
9,411,Article 83 - General conditions for imposing administrative fines,10,8,0.457,-4.68,2


P1 context: ['Article 84 - Penalties', 'Article 83 - General conditions for imposing administrative fines', 'Recital (148)']
P2 context: ['Recital (149)', 'Recital (148)', 'Recital (150)']

--- Answer 1 (baseline) ---
Based on the provided context, the maximum penalty depends on the specific violation and whether administrative fines or other penalties apply:

1. **Administrative Fines:** 
   * For infringements under Article 83(4), fines can be up to **10 000 000 EUR**, or up to **2% of the total worldwide annual turnover** of the preceding financial year for an undertaking, whichever is higher (Article 83(4)).
   * For infringements under Article 83(5), fines can be up to **20 000 000 EUR**, or up to **4% of the total worldwide annual turnover** of the preceding financial year, whichever is higher (Article 83(5)). 

2. **Other Penalties:** Member States must set rules for other effective, proportionate, and dissuasive penalties, particularly for infringements not subject to administr

...

P1 context: ['Recital (87)', 'Article 33 - Notification of a personal data breach to the supervisory authority', 'Recital (88)']
P2 context: ['Recital (87)', 'Recital (85)', 'Article 34 - Communication of a personal data breach to the data subject']

--- Answer 1 (baseline) ---
Based on the provided passages, there is not enough information to fully answer whether a company *always* has to inform affected people (data subjects) about a personal data breach. 

What the context provides:
* Recital (87) mentions that measures should be implemented to establish immediately whether a breach occurred and to "inform promptly the supervisory authority and the data subject," taking into account the nature, gravity, consequences, and adverse effects of the breach. 
* Article 33 specifies the rules for notifying the *supervisory authority*, stating notification is required "unless the personal data breach is unlikely to result in a risk to the rights and freedoms of natural persons." 

What is mi

,chunk_id,section,bi_rank,ce_rank,bi_score,ce_score,shift
0,135,Recital (107),1,1,0.772,5.27,0
1,129,Recital (102),2,9,0.703,-2.80,-7
2,130,Recital (103),3,2,0.700,4.56,1
3,144,Recital (114),4,4,0.699,3.77,0
4,341,Article 49 - Derogations for specific situations,5,8,0.690,-1.98,-3
5,338,Article 48 - Transfers or disclosures not authorised by Union law,6,5,0.682,0.49,1
6,325,Article 45 - Transfers on the basis of an adequacy decision,7,3,0.673,4.22,4
7,128,Recital (101),8,10,0.670,-3.12,-2
8,324,Article 44 - General principle for transfers,9,7,0.669,-1.81,2
9,141,Recital (112),10,6,0.666,0.43,4


P1 context: ['Recital (107)', 'Recital (102)', 'Recital (103)']
P2 context: ['Recital (107)', 'Recital (103)', 'Article 45 - Transfers on the basis of an adequacy decision']

--- Answer 1 (baseline) ---
Yes. According to Recital (103), when the Commission decides that a third country, a territory or specified sector within a third country, or an international organisation offers an adequate level of data protection, transfers of personal data to that third country or international organisation may take place "without the need to obtain any further authorisation."

--- Answer 2 (re-ranked) ---
Yes, no special approval or further authorisation is needed. 

According to **Article 45(1)**, when the Commission decides that a third country, a territory or specified sector within it, or an international organisation ensures an adequate level of protection, the transfer of personal data "shall not require any specific authorisation." 

Similarly, **Recital (103)** states that in cases where th

,chunk_id,section,bi_rank,ce_rank,bi_score,ce_score,shift
0,306,Article 38 - Position of the data protection officer,1,2,0.584,2.11,-1
1,304,Article 37 - Designation of the data protection officer,2,5,0.541,0.04,-3
2,124,Recital (97),3,1,0.532,3.39,2
3,307,Article 39 - Tasks of the data protection officer,4,4,0.512,0.59,0
4,305,Article 38 - Position of the data protection officer,5,3,0.509,1.69,2
5,111,Recital (87),6,7,0.491,-5.80,-1
6,390,Article 70 - Tasks of the Board,7,10,0.460,-9.26,-3
7,404,Article 80 - Representation of data subjects,8,8,0.458,-7.13,0
8,294,Article 34 - Communication of a personal data breach to the data subject,9,9,0.455,-8.43,0
9,303,Article 37 - Designation of the data protection officer,10,6,0.455,-0.91,4


P1 context: ['Article 38 - Position of the data protection officer', 'Article 37 - Designation of the data protection officer', 'Recital (97)']
P2 context: ['Recital (97)', 'Article 38 - Position of the data protection officer', 'Article 38 - Position of the data protection officer']

--- Answer 1 (baseline) ---
Based on the provided passages, there is not enough information to answer whether a data protection officer can be fired for doing their job. 

The provided texts cover:
- Confidentiality, conflicts of interest, and fulfilling other tasks (Article 38).
- The designation, qualifications, employment/service contract status, and publication of contact details of the data protection officer (Article 37).
- The requirement that data protection officers be in a position to perform duties independently (Recital 97).

The passages are missing specific rules or protections regarding dismissal, termination, or penalties relating to the performance of the data protection officer's tasks.


What the answers show:
- **E1 (exception), Art 34 #8 → #3.** With the baseline context (Recital 87, Article 33, Recital 88) Gemini says the passages do not cover informing data subjects. With the re-ranked context it answers "No" and lists the three Art 34(3) cases where informing people is not required. The baseline left out the exception completely.
- **K1 (keyword trap), Art 17 #4 → #3.** "Delete" vs "erasure". The bi-encoder's #2 was Article 33 on breach notification, which shares only general words like "controller" and "personal data". Answer 1 says Article 17 is missing; Answer 2 lists the grounds for erasure.
- **N11 (negation), Art 45 #7 → #3.** Both answers say "yes, no extra approval needed", but Answer 1 gets it only from Recital 103, while Answer 2 cites Article 45(1), the binding rule.
- **N13 (negation + synonym), Art 38 #5 → #3.** "Fired" vs "dismissed". Article 38 was already at #1 in the baseline, but it was the other chunk of Article 38, the one without the dismissal rule. Answer 1 says the passages do not cover dismissal; Answer 2 says no and quotes 38(3).
- **K3 (hurt), Art 83 #2 → #4.** "Penalty" pulled the cross-encoder towards Recitals 148-150, which talk about penalties in general. Answer 1 gives the correct EUR 20M / 4 % maximum; Answer 2 says the amount is not in the passages. So re-ranking is not always an improvement.

Because the prompt forbids outside knowledge, the baseline's failures show up as "not enough information" or a missing exception, not as made-up rules.

In [19]:
# Save the sweep, the summary metrics and, for each selected question, ranks, contexts and both answers
os.makedirs("results", exist_ok=True)
sweep.to_csv("results/failure_sweep.csv", index=False)
summary.to_csv("results/retrieval_summary.csv")
with open("results/failure_answers.json", "w") as f:
    json.dump({qid: {"query": r["query"],
                     "p1_context": [chunks_df.at[c, "section"] for c in r["p1_ids"]],
                     "p2_context": [chunks_df.at[c, "section"] for c in r["p2_ids"]],
                     "answer1": r["answer1"], "answer2": r["answer2"], "llm": r["model"],
                     "rank_table": r["table"].to_dict(orient="records")}
               for qid, r in failure_results.items()}, f, indent=2, default=str)
print("saved to results/")

saved to results/


## 5. Latency

Each retrieval step is timed on its own with `time.perf_counter()`:
- query embedding (bi-encoder on one short text)
- FAISS search over the 434 chunk vectors
- cross-encoder on the 10 (query, chunk) pairs, plus sorting

Three warm-up queries run first and are not recorded, so one-time costs like loading weights into memory do not count. Then all 29 questions are run 10 times each.
The Gemini call is timed separately on 3 questions. Most of that time is network and Google's servers, not my pipeline.

In [20]:
import platform
print("machine:", platform.platform(), "| processor:", platform.processor(), "| torch threads:", torch.get_num_threads(), "| device:", DEVICE)

latency_queries = [q["query"] for q in FAILURE_QUERIES]

for q in latency_queries[:3]:                    # warm-up, not recorded
    rerank(q, baseline_retrieve(q))

N_REPEATS = 10
records = []
for rep in range(N_REPEATS):
    for q in latency_queries:
        t0 = time.perf_counter()
        q_emb = bi_encoder.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        t1 = time.perf_counter()
        scores, ids = index.search(q_emb, TOP_K)
        t2 = time.perf_counter()
        pairs = [(q, chunks_df.at[cid, "text"]) for cid in ids[0]]
        ce_scores = cross_encoder.predict(pairs, batch_size=TOP_K, show_progress_bar=False)
        order = np.argsort(-ce_scores)           # the re-sort step of Pipeline 2
        t3 = time.perf_counter()
        records.append({"query embedding": (t1 - t0) * 1000,
                        "FAISS search (434 chunks)": (t2 - t1) * 1000,
                        "cross-encoder (10 pairs)": (t3 - t2) * 1000})

lat = pd.DataFrame(records)
lat["Pipeline 1 retrieval total"] = lat["query embedding"] + lat["FAISS search (434 chunks)"]
lat["Pipeline 2 retrieval total"] = lat["Pipeline 1 retrieval total"] + lat["cross-encoder (10 pairs)"]

latency_table = pd.DataFrame({
    "mean (ms)": lat.mean(),
    "median (ms)": lat.median(),
    "p95 (ms)": lat.quantile(0.95),          # 95 % of runs were faster than this
}).round(2)
print(f"{len(lat)} timed runs ({len(latency_queries)} queries x {N_REPEATS})")
latency_table

machine: macOS-26.6.2-arm64-arm-64bit | processor: arm | torch threads: 4 | device: cpu
290 timed runs (29 queries x 10)


,mean (ms),median (ms),p95 (ms)
query embedding,11.67,10.91,15.74
FAISS search (434 chunks),0.05,0.04,0.06
cross-encoder (10 pairs),163.69,157.88,205.44
Pipeline 1 retrieval total,11.72,10.95,15.87
Pipeline 2 retrieval total,175.41,168.99,220.91


In [21]:
# Gemini time: 3 questions, each once with the Pipeline 1 context and once with the Pipeline 2 context.
# use_cache=False makes real API calls (otherwise this would only time reading the saved answer).
llm_times = []
for q in latency_queries[:3]:
    cands = baseline_retrieve(q)
    reranked = rerank(q, cands)
    for name, ids in [("P1", cands["chunk_id"].tolist()[:TOP_N]), ("P2", reranked["chunk_id"].tolist()[:TOP_N])]:
        t0 = time.perf_counter()
        call_llm(build_prompt(q, ids), use_cache=False)
        llm_times.append({"pipeline": name, "model": LAST_MODEL, "llm_ms": (time.perf_counter() - t0) * 1000})
print("LLM calls timed with:", ", ".join(sorted({x["model"] for x in llm_times})))
llm_df = pd.DataFrame(llm_times)
llm_median = llm_df["llm_ms"].median()

# end-to-end = median retrieval time + median Gemini time, per pipeline
end_to_end = pd.DataFrame({
    "retrieval median (ms)": [latency_table.loc["Pipeline 1 retrieval total", "median (ms)"],
                              latency_table.loc["Pipeline 2 retrieval total", "median (ms)"]],
    "LLM median (ms)": [llm_df[llm_df.pipeline == "P1"]["llm_ms"].median(),
                        llm_df[llm_df.pipeline == "P2"]["llm_ms"].median()],
}, index=["Pipeline 1 (baseline)", "Pipeline 2 (re-ranked)"])
end_to_end["end-to-end (ms)"] = end_to_end.sum(axis=1)
end_to_end["re-ranking share of end-to-end"] = (latency_table.loc["cross-encoder (10 pairs)", "median (ms)"]
                                                / end_to_end["end-to-end (ms)"]).map("{:.1%}".format)
end_to_end.loc["Pipeline 1 (baseline)", "re-ranking share of end-to-end"] = "-"
print(f"index build (one-off): {index_build_seconds:.2f} s for {index.ntotal} chunks")
end_to_end.round(1)

LLM calls timed with: gemini-3.8-flash
index build (one-off): 4.70 s for 434 chunks


,retrieval median (ms),LLM median (ms),end-to-end (ms),re-ranking share of end-to-end
Pipeline 1 (baseline),11.0,4516.1,4527.1,-
Pipeline 2 (re-ranked),169.0,4248.4,4417.3,3.6%


Pipeline 1 retrieval takes about 11 ms, almost all of it embedding the query; the FAISS search itself is well under 1 ms. Re-ranking 10 pairs adds about 160 ms, so retrieval in Pipeline 2 is roughly 15 times slower.
Next to a Gemini call of about 4 to 4.5 seconds, though, that extra time is under 4 % of the end-to-end time. Gemini's time also changes a lot from call to call, so the LLM column should not be read as a difference between the pipelines.

## 6. Scaling: what if the cross-encoder scored every chunk?

The bi-encoder encodes every chunk once, ahead of time. For a new question it only encodes the question and compares one vector with the stored ones.
The cross-encoder cannot prepare anything in advance, because its score comes from reading the question and the chunk together. Every chunk needs a full pass through the model, for every question.

Below, the cross-encoder scores all 434 chunks for one question, and that time per pair is used to estimate bigger collections.
I also run the full ranking for the 10 questions whose gold chunk never made the bi-encoder's top 10, to see if the cross-encoder would have found them.

In [22]:
scale_query = "Does a company always have to inform affected people about a data breach?"
all_pairs = [(scale_query, t) for t in chunks_df["text"]]      # the question paired with every chunk

full_runs = []
for _ in range(3):
    t0 = time.perf_counter()
    cross_encoder.predict(all_pairs, batch_size=32, show_progress_bar=False)
    full_runs.append((time.perf_counter() - t0) * 1000)
full_ce_ms = float(np.median(full_runs))
per_pair_ms = full_ce_ms / len(all_pairs)

search_ms = latency_table.loc["FAISS search (434 chunks)", "median (ms)"]
embed_ms = latency_table.loc["query embedding", "median (ms)"]

# Searching only 434 vectors is mostly fixed overhead, so the search cost per vector is measured on
# 100,000 random unit vectors of the same size (384). Random values are fine: search time does not depend on content.
rng = np.random.default_rng(0)
synthetic = rng.standard_normal((100_000, chunk_embeddings.shape[1])).astype("float32")
synthetic /= np.linalg.norm(synthetic, axis=1, keepdims=True)
big_index = faiss.IndexFlatIP(synthetic.shape[1])
big_index.add(synthetic)
q_emb = bi_encoder.encode([scale_query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
big_index.search(q_emb, TOP_K)                    # warm-up
t0 = time.perf_counter()
for _ in range(20):
    big_index.search(q_emb, TOP_K)
search_100k_ms = (time.perf_counter() - t0) * 1000 / 20
per_vector_ms = search_100k_ms / 100_000
del synthetic, big_index                          # free the memory

print(f"cross-encoder over all {len(all_pairs)} chunks: {full_ce_ms:.0f} ms  ({per_pair_ms:.2f} ms per pair)")
print(f"bi-encoder for the same query: {embed_ms:.1f} ms embedding + {search_ms:.2f} ms search")
print(f"FAISS flat search over 100,000 vectors: {search_100k_ms:.1f} ms")

def fmt(ms):
    """Show a time in ms, s, min or h, whichever is readable."""
    if ms < 1000:     return f"{ms:.1f} ms"
    if ms < 60_000:   return f"{ms / 1000:.1f} s"
    if ms < 3.6e6:    return f"{ms / 60_000:.1f} min"
    return f"{ms / 3.6e6:.1f} h"

# Estimated time per question: the cross-encoder grows with (number of chunks x time per pair);
# the bi-encoder is one query embedding plus one dot product per stored vector
rows = []
for n in [434, 10_000, 100_000, 1_000_000]:
    rows.append({
        "chunks": f"{n:,}",
        "cross-encoder on all chunks": fmt(per_pair_ms * n),
        "bi-encoder (exact flat search)": fmt(embed_ms + max(search_ms, per_vector_ms * n)),
        "bi-encoder top 10 + cross-encoder rerank": fmt(embed_ms + max(search_ms, per_vector_ms * n) + per_pair_ms * TOP_K),
    })
scaling_table = pd.DataFrame(rows)
scaling_table

cross-encoder over all 434 chunks: 4712 ms  (10.86 ms per pair)
bi-encoder for the same query: 10.9 ms embedding + 0.04 ms search
FAISS flat search over 100,000 vectors: 3.3 ms


,chunks,cross-encoder on all chunks,bi-encoder (exact flat search),bi-encoder top 10 + cross-encoder rerank
0,434,4.7 s,10.9 ms,119.5 ms
1,"10,000",1.8 min,11.2 ms,119.8 ms
2,"100,000",18.1 min,14.2 ms,122.8 ms
3,"1,000,000",3.0 h,44.2 ms,152.8 ms


In [23]:
# For the questions where the gold chunk was not in the bi-encoder's top 10:
# where does it rank if each model ranks all 434 chunks?
missed = sweep[sweep["outcome"] == "not in top 10"]["qid"].tolist()
full_rows = []
for qid in missed:
    q = next(x for x in FAILURE_QUERIES if x["qid"] == qid)
    scores = cross_encoder.predict([(q["query"], t) for t in chunks_df["text"]], batch_size=32, show_progress_bar=False)
    full_order = chunks_df["chunk_id"].values[np.argsort(-scores)].tolist()          # all chunks, best first
    bi_scores = chunk_embeddings @ bi_encoder.encode([q["query"]], normalize_embeddings=True)[0]   # cosine with every chunk
    bi_order = chunks_df["chunk_id"].values[np.argsort(-bi_scores)].tolist()
    full_rows.append({"qid": qid, "query": q["query"],
                      "gold rank, bi-encoder over all chunks": gold_rank(bi_order, q["gold_ids"]),
                      "gold rank, cross-encoder over all chunks": gold_rank(full_order, q["gold_ids"])})
full_rank_df = pd.DataFrame(full_rows)
full_rank_df

,qid,query,"gold rank, bi-encoder over all chunks","gold rank, cross-encoder over all chunks"
0,K2,Can my job application be rejected purely by a computer program without a human looking at it?,20,2
1,E2,Is a company always required to erase my data when I ask?,11,4
2,E3,Does the GDPR apply to me keeping my friends' phone numbers on my personal phone?,50,30
3,E5,Does every non-EU company targeting EU customers need to appoint a representative in the EU?,109,12
4,N2,When can sensitive data not be processed even if the person consents?,17,11
5,N4,Can a processor hire a subcontractor without the controller's approval?,14,3
6,D2,When is a data protection impact assessment mandatory?,11,13
7,N7,What types of personal data processing does the GDPR not cover?,31,23
8,N8,When does a company not need to tell me that it collected my data from someone else?,27,21
9,N12,Which processing are data protection authorities not allowed to supervise?,132,42


Scoring all 434 chunks with the cross-encoder takes about 4.7 s for a single question, already longer than the Gemini call, and this is just one document. At about 11 ms per pair, 1 million chunks would take around 3 hours per question. The bi-encoder with a top-10 re-rank stays under 200 ms even at that size, because the chunk vectors are computed once and only the question is encoded at query time.

The second table shows that a full cross-encoder ranking would have brought 3 of the 10 missed chunks into its top 4 (K2, E2, N4), so a bigger candidate list (larger k) would help some of these questions. For others (E3, N7, N8, N12), both models rank the gold chunk far down, so the problem there is the models, not only the first stage. In practice this is the trade-off: a larger k gives better recall but re-ranking time grows linearly with k.

In [24]:
latency_table.to_csv("results/latency_retrieval.csv")
end_to_end.to_csv("results/latency_end_to_end.csv")
scaling_table.to_csv("results/scaling_estimate.csv", index=False)
full_rank_df.to_csv("results/full_cross_encoder_ranks.csv", index=False)
print("saved to results/")

saved to results/
